# Metrics Comparison

This notebook loads the saved evaluation outputs for `Fluo-N2DL-HeLa / 02`, compares the original methods with the consensus variants, and can export each table as a white-background PNG.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from PIL import Image, ImageDraw, ImageFont

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.options.display.max_columns = None

SEQUENCE_DIR = Path("/group/jug/Sheida/PyTr2d/outputs/Fluo-N2DL-HeLa/02")
CONSENSUS_DIR = SEQUENCE_DIR / "consensus_embedseg_stardist"
EXPORT_DIR = CONSENSUS_DIR / "table_exports"

SORT_BY = "OP_CTB"
BEST_COLOR = "#93c47d"
WORST_COLOR = "#e06666"
WHITE = "#ffffff"
GRID_COLOR = "#cfcfcf"
TEXT_COLOR = "#111111"

SEQUENCE_DIR, CONSENSUS_DIR, EXPORT_DIR


(PosixPath('/group/jug/Sheida/PyTr2d/outputs/Fluo-N2DL-HeLa/02'),
 PosixPath('/group/jug/Sheida/PyTr2d/outputs/Fluo-N2DL-HeLa/02/consensus_embedseg_stardist'),
 PosixPath('/group/jug/Sheida/PyTr2d/outputs/Fluo-N2DL-HeLa/02/consensus_embedseg_stardist/table_exports'))

In [2]:
def parse_number(value):
    if isinstance(value, (int, float)):
        return value

    try:
        number = float(value)
    except (TypeError, ValueError):
        return value

    return int(number) if number.is_integer() else number


def parse_ctc_metric_map(metric_map):
    if not metric_map:
        return {}

    if len(metric_map) != 1:
        raise ValueError(f"Expected a single semicolon metric entry, got: {metric_map}")

    header_row, value_row = next(iter(metric_map.items()))
    headers = [item.strip() for item in header_row.split(";")]
    values = [item.strip() for item in value_row.split(";")]

    if len(headers) != len(values):
        raise ValueError(f"Header/value length mismatch in CTC metric block: {metric_map}")

    return {header: parse_number(value) for header, value in zip(headers, values)}


def load_json(path):
    return json.loads(path.read_text())


def load_single_metrics_from_text(path):
    payload = {"ctc_evaluation": {}, "summary_metrics": {}}
    section = None
    lines = path.read_text().splitlines()

    i = 0
    while i < len(lines):
        line = lines[i]
        stripped = line.strip()
        indent = len(line) - len(line.lstrip(" "))

        if indent == 0 and stripped.endswith(":"):
            section = stripped[:-1]
        elif section == "ctc_evaluation" and indent == 2 and stripped == "metrics:" and i + 1 < len(lines):
            header_row, value_row = lines[i + 1].strip().split(": ", 1)
            payload["ctc_evaluation"]["metrics"] = {header_row: value_row}
            i += 1
        elif section == "summary_metrics" and indent == 2 and ": " in stripped:
            key, value = stripped.split(": ", 1)
            payload["summary_metrics"][key] = parse_number(value)

        i += 1

    return {key: value for key, value in payload.items() if value}


def load_variant_comparison_from_text(path):
    payload = {}
    current_method = None
    section = None
    lines = path.read_text().splitlines()

    i = 0
    while i < len(lines):
        line = lines[i]
        stripped = line.strip()
        indent = len(line) - len(line.lstrip(" "))

        if indent == 0 and stripped.endswith(":"):
            current_method = stripped[:-1]
            payload[current_method] = {
                "ctc_evaluation": {},
                "legacy_gt_metrics": {},
                "summary_metrics": {},
            }
            section = None
        elif current_method and indent == 2 and stripped.endswith(":"):
            section = stripped[:-1]
        elif current_method and section == "ctc_evaluation" and indent == 4 and stripped == "metrics:" and i + 1 < len(lines):
            header_row, value_row = lines[i + 1].strip().split(": ", 1)
            payload[current_method]["ctc_evaluation"]["metrics"] = {header_row: value_row}
            i += 1
        elif current_method and section in {"legacy_gt_metrics", "summary_metrics"} and indent == 4 and ": " in stripped:
            key, value = stripped.split(": ", 1)
            payload[current_method][section][key] = parse_number(value)

        i += 1

    return {
        method: {key: value for key, value in method_payload.items() if value}
        for method, method_payload in payload.items()
    }


def load_consensus_payloads(consensus_dir):
    comparison_json = consensus_dir / "variant_comparison.json"
    comparison_text = consensus_dir / "variant_comparison.txt"

    if comparison_json.exists():
        return load_json(comparison_json)
    if comparison_text.exists():
        return load_variant_comparison_from_text(comparison_text)

    raise FileNotFoundError(f"Could not find variant comparison files in {consensus_dir}")


def load_single_method_payload(method_dir):
    metrics_json = method_dir / "metrics.json"
    metrics_text = method_dir / "metrics.txt"

    if metrics_json.exists():
        return load_json(metrics_json)
    if metrics_text.exists():
        return load_single_metrics_from_text(metrics_text)

    raise FileNotFoundError(f"Could not find metrics.json or metrics.txt in {method_dir}")


def build_record(method, payload, run_type, source_path):
    record = {
        "run_type": run_type,
        "method": method,
        "source_path": str(source_path),
    }
    record.update(parse_ctc_metric_map(payload.get("ctc_evaluation", {}).get("metrics", {})))
    record.update(payload.get("summary_metrics", {}))
    record.update({f"legacy_{key}": value for key, value in payload.get("legacy_gt_metrics", {}).items()})
    return record


def load_tables(sequence_dir, consensus_dir):
    base_records = []
    for method_dir in sorted(sequence_dir.iterdir()):
        if not method_dir.is_dir() or method_dir.name.startswith("consensus_"):
            continue
        if not ((method_dir / "metrics.json").exists() or (method_dir / "metrics.txt").exists()):
            continue
        base_records.append(
            build_record(
                method=method_dir.name,
                payload=load_single_method_payload(method_dir),
                run_type="single",
                source_path=method_dir,
            )
        )

    consensus_records = []
    consensus_payloads = load_consensus_payloads(consensus_dir)
    for method, payload in consensus_payloads.items():
        consensus_records.append(
            build_record(
                method=method,
                payload=payload,
                run_type="consensus",
                source_path=consensus_dir / method,
            )
        )

    base_df = pd.DataFrame(base_records)
    consensus_df = pd.DataFrame(consensus_records)
    all_df = pd.concat([base_df, consensus_df], ignore_index=True, sort=False)
    return base_df, consensus_df, all_df


ZERO_DECIMAL_COLUMNS = {
    "Valid",
    "track_count",
    "division_count",
    "frame_count",
    "frame_object_count_min",
    "frame_object_count_max",
    "AOGM",
    "AOGM_0",
    "AOGM_NS",
    "AOGM_FN",
    "AOGM_FP",
    "AOGM_ED",
    "AOGM_EA",
    "AOGM_EC",
}


def format_value(value, column):
    if pd.isna(value):
        return "-"
    if isinstance(value, (int, float)):
        if column in ZERO_DECIMAL_COLUMNS:
            return f"{value:.0f}"
        return f"{value:.4f}"
    return str(value)


def styled_table(df, higher_is_better=None, lower_is_better=None):
    higher_is_better = [column for column in (higher_is_better or []) if column in df.columns]
    lower_is_better = [column for column in (lower_is_better or []) if column in df.columns]

    formatters = {}
    for column in df.columns:
        if pd.api.types.is_numeric_dtype(df[column]):
            if column in ZERO_DECIMAL_COLUMNS:
                formatters[column] = "{:.0f}".format
            else:
                formatters[column] = "{:.4f}".format

    styler = df.style.format(formatters, na_rep="-")
    if higher_is_better:
        styler = styler.highlight_max(subset=higher_is_better, color=BEST_COLOR)
    if lower_is_better:
        styler = styler.highlight_min(subset=lower_is_better, color=WORST_COLOR)
    return styler


def load_font(size, bold=False):
    candidates = [
        "DejaVuSans-Bold.ttf" if bold else "DejaVuSans.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    ]
    for candidate in candidates:
        try:
            return ImageFont.truetype(candidate, size=size)
        except OSError:
            continue
    return ImageFont.load_default()


def text_size(draw, text, font):
    left, top, right, bottom = draw.textbbox((0, 0), str(text), font=font)
    return right - left, bottom - top


def save_table_png(df, output_path, higher_is_better=None, lower_is_better=None, title=None):
    export_df = df.reset_index()
    export_df.columns = [str(column) for column in export_df.columns]

    higher_is_better = [column for column in (higher_is_better or []) if column in export_df.columns]
    lower_is_better = [column for column in (lower_is_better or []) if column in export_df.columns]

    formatted_df = export_df.copy()
    for column in formatted_df.columns:
        formatted_df[column] = formatted_df[column].map(lambda value, col=column: format_value(value, col))

    highlight_map = {}
    for column in higher_is_better:
        if not pd.api.types.is_numeric_dtype(export_df[column]):
            continue
        valid_values = export_df[column].dropna()
        if valid_values.empty:
            continue
        best_value = valid_values.max()
        column_index = export_df.columns.get_loc(column)
        for row_index, value in enumerate(export_df[column].tolist()):
            if pd.notna(value) and value == best_value:
                highlight_map[(row_index, column_index)] = BEST_COLOR

    for column in lower_is_better:
        if not pd.api.types.is_numeric_dtype(export_df[column]):
            continue
        valid_values = export_df[column].dropna()
        if valid_values.empty:
            continue
        worst_value = valid_values.min()
        column_index = export_df.columns.get_loc(column)
        for row_index, value in enumerate(export_df[column].tolist()):
            if pd.notna(value) and value == worst_value:
                highlight_map[(row_index, column_index)] = WORST_COLOR

    header_font = load_font(size=18, bold=True)
    body_font = load_font(size=16, bold=False)
    padding_x = 12
    padding_y = 8
    outer_padding = 20

    scratch = Image.new("RGB", (1, 1), WHITE)
    scratch_draw = ImageDraw.Draw(scratch)

    headers = list(formatted_df.columns)
    rows = formatted_df.values.tolist()

    column_widths = []
    for column_index, header in enumerate(headers):
        header_width, _ = text_size(scratch_draw, header, header_font)
        cell_width = header_width
        for row in rows:
            row_width, _ = text_size(scratch_draw, row[column_index], body_font)
            cell_width = max(cell_width, row_width)
        column_widths.append(cell_width + 2 * padding_x)

    _, header_text_height = text_size(scratch_draw, "Ag", header_font)
    _, body_text_height = text_size(scratch_draw, "Ag", body_font)
    header_height = header_text_height + 2 * padding_y
    row_height = body_text_height + 2 * padding_y
    title_height = 0
    if title:
        _, title_text_height = text_size(scratch_draw, title, header_font)
        title_height = title_text_height + 2 * padding_y

    image_width = int(sum(column_widths) + 2 * outer_padding)
    image_height = int(title_height + header_height + len(rows) * row_height + 2 * outer_padding)

    image = Image.new("RGB", (image_width, image_height), WHITE)
    draw = ImageDraw.Draw(image)

    current_y = outer_padding
    if title:
        draw.text((outer_padding, current_y), title, fill=TEXT_COLOR, font=header_font)
        current_y += title_height

    current_x = outer_padding
    for column_index, header in enumerate(headers):
        cell_box = [current_x, current_y, current_x + column_widths[column_index], current_y + header_height]
        draw.rectangle(cell_box, fill=WHITE, outline=GRID_COLOR, width=1)
        text_width, text_height = text_size(draw, header, header_font)
        text_x = current_x + (column_widths[column_index] - text_width) / 2
        text_y = current_y + (header_height - text_height) / 2 - 1
        draw.text((text_x, text_y), header, fill=TEXT_COLOR, font=header_font)
        current_x += column_widths[column_index]

    current_y += header_height
    for row_index, row in enumerate(rows):
        current_x = outer_padding
        for column_index, value in enumerate(row):
            fill_color = highlight_map.get((row_index, column_index), WHITE)
            cell_box = [current_x, current_y, current_x + column_widths[column_index], current_y + row_height]
            draw.rectangle(cell_box, fill=fill_color, outline=GRID_COLOR, width=1)

            text_width, text_height = text_size(draw, value, body_font)
            is_numeric = pd.api.types.is_numeric_dtype(export_df.iloc[:, column_index])
            if is_numeric:
                text_x = current_x + (column_widths[column_index] - text_width) / 2
            else:
                text_x = current_x + padding_x
            text_y = current_y + (row_height - text_height) / 2 - 1
            draw.text((text_x, text_y), value, fill=TEXT_COLOR, font=body_font)

            current_x += column_widths[column_index]
        current_y += row_height

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    image.save(output_path)
    return output_path


In [3]:
base_df, consensus_df, all_df = load_tables(SEQUENCE_DIR, CONSENSUS_DIR)

loaded_sources = (
    all_df[["run_type", "method", "source_path"]]
    .sort_values(["run_type", "method"])
    .reset_index(drop=True)
)

loaded_sources


,run_type,method,source_path
0,consensus,embedseg,/group/jug/Sheida/PyTr2d/outputs/Fluo-N2DL-HeL...
1,consensus,intersection,/group/jug/Sheida/PyTr2d/outputs/Fluo-N2DL-HeL...
2,consensus,stardist,/group/jug/Sheida/PyTr2d/outputs/Fluo-N2DL-HeL...
3,consensus,union,/group/jug/Sheida/PyTr2d/outputs/Fluo-N2DL-HeL...
4,single,embedseg,/group/jug/Sheida/PyTr2d/outputs/Fluo-N2DL-HeL...
5,single,stardist,/group/jug/Sheida/PyTr2d/outputs/Fluo-N2DL-HeL...


In [4]:
score_columns = ["OP_CTB", "OP_CSB", "TRA", "DET", "SEG", "LNK", "CT", "TF", "CCA"]
lower_is_better_columns = ["AOGM", "AOGM_FN", "AOGM_FP"]
context_columns = ["track_count", "division_count", "frame_object_count_mean", "common_tracklet_coverage"]

compact_columns = [
    column
    for column in score_columns + lower_is_better_columns + context_columns
    if column in all_df.columns
]

consensus_view = (
    consensus_df[["method", *compact_columns]]
    .sort_values(SORT_BY, ascending=False)
    .set_index("method")
)

all_view = (
    all_df[["run_type", "method", *compact_columns]]
    .sort_values([SORT_BY, "run_type", "method"], ascending=[False, True, True])
    .set_index(["run_type", "method"])
)

full_metric_columns = [
    column
    for column in [
        "Valid",
        "DET",
        "SEG",
        "TRA",
        "CT",
        "TF",
        "CCA",
        "LNK",
        "OP_CSB",
        "OP_CTB",
        "AOGM",
        "AOGM_0",
        "AOGM_NS",
        "AOGM_FN",
        "AOGM_FP",
        "AOGM_ED",
        "AOGM_EA",
        "AOGM_EC",
        "track_count",
        "division_count",
        "frame_count",
        "frame_object_count_min",
        "frame_object_count_mean",
        "frame_object_count_max",
        "common_tracklet_coverage",
        "legacy_vertex_precision",
        "legacy_vertex_recall",
        "legacy_vertex_f1",
        "legacy_link_precision",
        "legacy_link_recall",
        "legacy_link_f1",
        "legacy_CT",
        "legacy_TF",
        "legacy_BC(0)",
        "legacy_BIO",
    ]
    if column in all_df.columns
]

full_view = (
    all_df[["run_type", "method", *full_metric_columns]]
    .sort_values([SORT_BY, "run_type", "method"], ascending=[False, True, True])
    .set_index(["run_type", "method"])
)

display(
    styled_table(
        consensus_view,
        higher_is_better=score_columns,
        lower_is_better=lower_is_better_columns,
    ).set_caption("Consensus variants")
)

display(
    styled_table(
        all_view,
        higher_is_better=score_columns,
        lower_is_better=lower_is_better_columns,
    ).set_caption("Original single-method runs and consensus variants")
)

display(full_view)


,OP_CTB,OP_CSB,TRA,DET,SEG,LNK,CT,TF,CCA,AOGM,AOGM_FN,AOGM_FP,track_count,division_count,frame_object_count_mean,common_tracklet_coverage
method,,,,,,,,,,,,,,,,
embedseg,0.9214,0.9224,0.9706,0.9726,0.8722,0.9575,0.3570,0.8876,0.4823,8571,183,4548,1186,874,322.4674,1.0000
intersection,0.9212,0.9222,0.9705,0.9725,0.8719,0.9574,0.3559,0.8873,0.4823,8599,185,4550,1186,874,322.4674,1.0000
union,0.9211,0.9221,0.9707,0.9726,0.8716,0.9576,0.3570,0.8880,0.4823,8558,182,4547,1186,874,322.4674,1.0000
stardist,0.9210,0.9220,0.9706,0.9726,0.8713,0.9575,0.3570,0.8880,0.4823,8572,183,4548,1186,874,322.4674,1.0000


Valid       DET       SEG       TRA        CT  \
run_type  method                                                        
single    embedseg          1  0.972864  0.875005  0.970824  0.349869   
consensus embedseg          1  0.972589  0.872181  0.970642  0.356989   
          intersection      1  0.972502  0.871936  0.970547  0.355914   
          union             1  0.972632  0.871602  0.970685  0.356989   
          stardist          1  0.972589  0.871346  0.970637  0.356989   
single    stardist          1  0.964752  0.853122  0.961340  0.353008   

                              TF       CCA       LNK    OP_CSB    OP_CTB  \
run_type  method                                                           
single    embedseg      0.889081  0.438136  0.957088  0.923934  0.922914   
consensus embedseg      0.887603  0.482283  0.957539  0.922385  0.921411   
          intersection  0.887291  0.482283  0.957380  0.922219  0.921241   
          union         0.888001  0.482283  0.957578  0.922117  0.921144   
          stardist      0.888001  0.482283  0.957499  0.921967  0.920992   
single    stardist      0.871176  0.508502  0.938361  0.908937  0.907231   

                           AOGM  AOGM_0  AOGM_NS  AOGM_FN  AOGM_FP  AOGM_ED  \
run_type  method                                                              
single    embedseg       8518.0  291952      118      181     4498      164   
consensus embedseg       8571.0  291952      118      183     4548      151   
          intersection   8599.0  291952      118      185     4550      151   
          union          8558.5  291952      118      182     4547      151   
          stardist       8572.5  291952      118      183     4548      151   
single    stardist      11287.0  291952      198      500     2970      197   

                        AOGM_EA  AOGM_EC  track_count  division_count  \
run_type  method                                                        
single    embedseg          854      175       1241.0           932.0   
consensus embedseg          858      165       1186.0           874.0   
          intersection      862      165       1186.0           874.0   
          union             857      165       1186.0           874.0   
          stardist          859      165       1186.0           874.0   
single    stardist         1302      177       1088.0           786.0   

                        frame_count  frame_object_count_min  \
run_type  method                                              
single    embedseg             92.0                   150.0   
consensus embedseg             92.0                   150.0   
          intersection         92.0                   150.0   
          union                92.0                   150.0   
          stardist             92.0                   150.0   
single    stardist             92.0                   141.0   

                        frame_object_count_mean  frame_object_count_max  \
run_type  method                                                          
single    embedseg                   321.945652                   414.0   
consensus embedseg                   322.467391                   416.0   
          intersection               322.467391                   416.0   
          union                      322.467391                   416.0   
          stardist                   322.467391                   416.0   
single    stardist                   301.000000                   388.0   

                        common_tracklet_coverage  legacy_vertex_precision  \
run_type  method                                                            
single    embedseg                           NaN                      NaN   
consensus embedseg                           1.0                 0.000843   
          intersection                       1.0                 0.000910   
          union                              1.0                 0.000843   
          stardist                          

In [5]:
exported_paths = {
    "consensus_variants": save_table_png(
        consensus_view,
        EXPORT_DIR / "consensus_variants.png",
        higher_is_better=score_columns,
        lower_is_better=lower_is_better_columns,
        title="Consensus variants",
    ),
    "all_methods": save_table_png(
        all_view,
        EXPORT_DIR / "all_methods.png",
        higher_is_better=score_columns,
        lower_is_better=lower_is_better_columns,
        title="Original single-method runs and consensus variants",
    ),
    "full_metrics": save_table_png(
        full_view,
        EXPORT_DIR / "full_metrics.png",
        higher_is_better=score_columns,
        lower_is_better=lower_is_better_columns,
        title="All available metrics",
    ),
}

pd.Series({name: str(path) for name, path in exported_paths.items()}, name="png_path")


consensus_variants    /group/jug/Sheida/PyTr2d/outputs/Fluo-N2DL-HeL...
all_methods           /group/jug/Sheida/PyTr2d/outputs/Fluo-N2DL-HeL...
full_metrics          /group/jug/Sheida/PyTr2d/outputs/Fluo-N2DL-HeL...
Name: png_path, dtype: str